# Citus Database Setup and SDK Demo (Single Node)

This notebook demonstrates how to set up a Citus database using a single-node docker container, populate the patch table with dummy data, and implement a Python SDK for interacting with the database. Worker node logic is omitted for single-node setup.

## 1. Install and Import Required Libraries

Install `psycopg` if not already installed, and import all required libraries for database interaction.

In [1]:
# Install psycopg if needed (uncomment if running in a new environment)
# !pip install psycopg[binary]

import psycopg
from psycopg.rows import dict_row
import random
import datetime
import base64
from db_client import CitusHeadClient
import os
import numpy as np
import io
from PIL import Image

In [2]:
# Import DB connection constants from constants.py
from constants import (
    CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD
)

In [3]:
# Set DB connection variables from constants (single-node)
DB_HOST = CITUS_HEAD_HOST
DB_PORT = CITUS_HEAD_PORT
DB_NAME = CITUS_HEAD_DB
DB_USER = CITUS_HEAD_USER
DB_PASSWORD = CITUS_HEAD_PASSWORD

In [4]:
NUM_PATCHES = 100000

## 0. Drop All Tables (Clean Start)

Drop all tables if they exist to ensure a clean setup.

In [5]:
head_client = CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)
head_client.drop_all_tables()
print("All tables dropped (if existed).")


All tables dropped (if existed).


## 2. Connect to Citus Node

Establish a connection to the Citus/Postgres node using psycopg. Store connection parameters securely (e.g., using environment variables).

In [6]:
# Use CitusHeadClient for connection
def get_head_connection():
    return CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).get_connection()

# Test connection
with get_head_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        print('Connected to:', cur.fetchone()['version'])

Connected to: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create Database Schema (Tables)

Create all tables as described in the technical design document, including distributed and reference tables. Use Citus distribution commands where required.

In [7]:
head_client.setup_schema()
print("Schema and distribution setup complete.")


Schema and distribution setup complete.


In [8]:
head_client.setup_triggers()


INSERT trigger function created on coordinator and workers.
UPDATE trigger function created on coordinator and workers.
Per-shard INSERT triggers installed on pred_patch_latest shards.
Per-shard INSERT triggers installed on pred_patch_last shards.
Per-shard UPDATE triggers installed on patch shards.

All per-shard triggers installed.


## 4. Verify Table Creation

Query the information schema to verify that all tables have been created successfully.

In [9]:
# List all tables in the public schema
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = [row['table_name'] for row in cur.fetchall()]
        print('Tables in public schema:', tables)

Tables in public schema: ['citus_schemas', 'citus_tables', 'confusion_matrix_l10', 'confusion_matrix_l11', 'confusion_matrix_l12', 'confusion_matrix_l8', 'confusion_matrix_l9', 'image', 'label_class', 'patch', 'pred_patch_last', 'pred_patch_latest', 'project', 'settings']


## 5. Insert Dummy Data into Patch Table

Generate and insert dummy data into the patch table, ensuring all required fields are populated and constraints are respected.

In [10]:

# Helper: Insert dummy project, image, label_class for FK constraints
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO project (project_name, description) VALUES (%s, %s) RETURNING project_id;", ('Demo Project', 'For dummy data'))
        project_id = cur.fetchone()['project_id']
        cur.execute("INSERT INTO image (project_id, name, image_path, upload_ts, base_mag, base_width, base_height, deepzoom_tilesize) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) RETURNING image_id;",
                    (project_id, 'Demo Image', '/tmp/demo.tif', datetime.datetime.now(), 20.0, 10000, 8000, 256))
        image_id = cur.fetchone()['image_id']
        cur.execute("INSERT INTO label_class (project_id, name, color_code, event_ts) VALUES (%s, %s, %s, %s) RETURNING label_class_id;",
                    (project_id, 'Tumor', '#FF0000', datetime.datetime.now()))
        label_class_id = cur.fetchone()['label_class_id']
        print(f"Inserted project_id={project_id}, image_id={image_id}, label_class_id={label_class_id}")

CONSTANT_IMAGE = bytes(128)  # 128 zero bytes used as a placeholder patch image

BATCH_SIZE = 500
JITTER = 8  # ±pixel range around the base color

def make_jitter_jpeg() -> bytes:
    base_color = np.random.randint(0, 256, (1, 1, 3), dtype=np.int16)
    jitter = np.random.randint(-JITTER, JITTER + 1, (32, 32, 3), dtype=np.int16)
    arr = np.clip(base_color + jitter, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr, mode='RGB')
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

for batch_start in range(0, NUM_PATCHES, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, NUM_PATCHES)
    batch = [
        (1000 + i, label_class_id, image_id, 20.0, make_jitter_jpeg())
        for i in range(batch_start, batch_end)
    ]
    head_client.bulk_insert_patches(batch)
    print(f"Inserted patches {batch_start + 1}–{batch_end}")

print(f"Done. {NUM_PATCHES} patches inserted.")

Inserted project_id=1, image_id=1, label_class_id=1
Inserted patches 1–500
Inserted patches 501–1000
Inserted patches 1001–1500
Inserted patches 1501–2000
Inserted patches 2001–2500
Inserted patches 2501–3000
Inserted patches 3001–3500
Inserted patches 3501–4000
Inserted patches 4001–4500
Inserted patches 4501–5000
Inserted patches 5001–5500
Inserted patches 5501–6000
Inserted patches 6001–6500
Inserted patches 6501–7000
Inserted patches 7001–7500
Inserted patches 7501–8000
Inserted patches 8001–8500
Inserted patches 8501–9000
Inserted patches 9001–9500
Inserted patches 9501–10000
Inserted patches 10001–10500
Inserted patches 10501–11000
Inserted patches 11001–11500
Inserted patches 11501–12000
Inserted patches 12001–12500
Inserted patches 12501–13000
Inserted patches 13001–13500
Inserted patches 13501–14000
Inserted patches 14001–14500
Inserted patches 14501–15000
Inserted patches 15001–15500
Inserted patches 15501–16000
Inserted patches 16001–16500
Inserted patches 16501–17000
Insert

In [11]:
# Check number of shards for the patch table and print row counts per shard, including empty shards
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        # Number of shards
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'patch' table: {num_shards}")
        # Row counts per shard, including empty
        cur.execute("""
            SELECT s.shardid, COALESCE(count(p.patch_id), 0) as row_count
            FROM pg_dist_shard s
            LEFT JOIN patch p ON get_shard_id_for_distribution_column('patch', p.patch_id) = s.shardid
            WHERE s.logicalrelid = 'patch'::regclass
            GROUP BY s.shardid
            ORDER BY s.shardid;
        """)
        rows = cur.fetchall()
        empty_count = 0
        for row in rows:
            print(f"Shard {row['shardid']}: {row['row_count']} rows")
            if row['row_count'] == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")

Number of shards for 'patch' table: 32
Shard 128852: 3066 rows
Shard 128853: 3055 rows
Shard 128854: 3099 rows
Shard 128855: 3248 rows
Shard 128856: 3111 rows
Shard 128857: 3095 rows
Shard 128858: 3131 rows
Shard 128859: 3128 rows
Shard 128860: 3140 rows
Shard 128861: 3180 rows
Shard 128862: 3197 rows
Shard 128863: 3129 rows
Shard 128864: 3135 rows
Shard 128865: 3083 rows
Shard 128866: 3109 rows
Shard 128867: 3193 rows
Shard 128868: 3132 rows
Shard 128869: 3061 rows
Shard 128870: 3007 rows
Shard 128871: 3167 rows
Shard 128872: 3084 rows
Shard 128873: 3120 rows
Shard 128874: 3200 rows
Shard 128875: 3145 rows
Shard 128876: 3029 rows
Shard 128877: 3191 rows
Shard 128878: 3186 rows
Shard 128879: 3107 rows
Shard 128880: 3138 rows
Shard 128881: 3135 rows
Shard 128882: 3081 rows
Shard 128883: 3118 rows
Empty shards: 0 out of 32


In [12]:
# Check number of shards for the confusion_matrix_ln table and print row counts per shard
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'confusion_matrix_ln'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'confusion_matrix_ln' table: {num_shards}")
        rows = cur.fetchall() if False else []
        cur.execute("""
            SELECT shardid FROM pg_dist_shard
            WHERE logicalrelid = 'confusion_matrix_ln'::regclass
            ORDER BY shardid;
        """)
        shard_ids = [row['shardid'] for row in cur.fetchall()]
        empty_count = 0
        for shard_id in shard_ids:
            cur.execute(f"SELECT count(*) FROM public.confusion_matrix_ln_{shard_id};")
            row_count = cur.fetchone()['count']
            print(f"Shard {shard_id}: {row_count} rows")
            if row_count == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")


UndefinedTable: relation "confusion_matrix_ln" does not exist
LINE 1: ... count(*) FROM pg_dist_shard WHERE logicalrelid = 'confusion...
                                                             ^

## 6. Verify Dummy Data in Patch Table

Query the patch table to confirm that dummy data has been inserted correctly.

In [ ]:
# Query and display dummy patch data
for row in head_client.fetch_patches(limit=10):
    print(row)

## 7. Implement db_client SDK: Head Node Level

Write Python classes and functions in `db_client.py` to interact with the database at the Citus head node level, including connection management and basic CRUD operations.

In [ ]:
# db_client.py will be implemented in the next step.
# Example usage for SDK will be shown after SDK implementation.

In [ ]:
# Example: Using db_client SDK (single-node)
# Uses constants.py for all connection parameters
head_client = CitusHeadClient()
print('Patches:', head_client.fetch_patches(limit=3))

# Insert a new patch (dummy data)
# patch_id = head_client.insert_patch(2000, 1, 1, 20.0, b'dummybytes')
# print('Inserted patch_id:', patch_id)

<!-- Worker node logic omitted for single-node setup -->